# 8. NumPy for NLP and computational linguistics

## Why this lesson matters

Text begins as strings, but most NLP models need numbers. NumPy helps us create token-id
sequences, count matrices, masks, and embedding calculations.

**Learning goals**

- build a tiny vocabulary;
- convert tokens to integer IDs;
- pad variable-length sequences;
- create attention-style masks;
- build a document-term matrix;
- work with embeddings and cosine similarity;
- perform simple nearest-neighbor search.


In [1]:
import numpy as np


## A tiny Persian corpus

This tokenizer is deliberately simple: it splits on spaces. Real Persian NLP needs careful
normalization and tokenization, but the numerical ideas remain the same.


In [2]:
sentences = [
    "من زبان فارسی را دوست دارم",
    "پردازش زبان طبیعی جذاب است",
    "من پردازش متن را دوست دارم",
]

tokenized = [sentence.split() for sentence in sentences]
print(tokenized)


[['من', 'زبان', 'فارسی', 'را', 'دوست', 'دارم'], ['پردازش', 'زبان', 'طبیعی', 'جذاب', 'است'], ['من', 'پردازش', 'متن', 'را', 'دوست', 'دارم']]


## Build a vocabulary

Reserve `0` for padding and `1` for unknown words.


In [3]:
special_tokens = ["<PAD>", "<UNK>"]
unique_words = sorted({word for sentence in tokenized for word in sentence})
vocabulary = special_tokens + unique_words
word_to_id = {word: index for index, word in enumerate(vocabulary)}

print("vocabulary size:", len(vocabulary))
print(word_to_id)


vocabulary size: 13
{'<PAD>': 0, '<UNK>': 1, 'است': 2, 'جذاب': 3, 'دارم': 4, 'دوست': 5, 'را': 6, 'زبان': 7, 'طبیعی': 8, 'فارسی': 9, 'متن': 10, 'من': 11, 'پردازش': 12}


## Convert words to IDs


In [4]:
encoded = [
    [word_to_id.get(word, word_to_id["<UNK>"]) for word in sentence]
    for sentence in tokenized
]

print(encoded)


[[11, 7, 9, 6, 5, 4], [12, 7, 8, 3, 2], [11, 12, 10, 6, 5, 4]]


The sequences have different lengths, so they cannot directly form a rectangular numerical
array. We pad shorter sequences with the `<PAD>` ID.


In [5]:
max_length = max(len(sequence) for sequence in encoded)
pad_id = word_to_id["<PAD>"]

padded = np.full((len(encoded), max_length), pad_id, dtype=np.int32)
for row, sequence in enumerate(encoded):
    padded[row, :len(sequence)] = sequence

print(padded)
print("shape:", padded.shape)


[[11  7  9  6  5  4]
 [12  7  8  3  2  0]
 [11 12 10  6  5  4]]
shape: (3, 6)


## Create a mask

`1` marks a real token and `0` marks padding. Transformer libraries use a similar idea for
attention masks.


In [6]:
attention_mask = (padded != pad_id).astype(np.int32)
print(attention_mask)


[[1 1 1 1 1 1]
 [1 1 1 1 1 0]
 [1 1 1 1 1 1]]


## Document-term matrix

Rows are documents, columns are vocabulary words, and each cell contains a word count.
We omit the two special tokens here.


In [7]:
lexical_words = vocabulary[2:]
lexical_to_column = {word: index for index, word in enumerate(lexical_words)}
document_term = np.zeros((len(tokenized), len(lexical_words)), dtype=np.int32)

for row, sentence in enumerate(tokenized):
    for word in sentence:
        document_term[row, lexical_to_column[word]] += 1

print("columns:", lexical_words)
print(document_term)


columns: ['است', 'جذاب', 'دارم', 'دوست', 'را', 'زبان', 'طبیعی', 'فارسی', 'متن', 'من', 'پردازش']
[[0 0 1 1 1 1 0 1 0 1 0]
 [1 1 0 0 0 1 1 0 0 0 1]
 [0 0 1 1 1 0 0 0 1 1 1]]


## Word-frequency totals across the corpus


In [8]:
corpus_counts = document_term.sum(axis=0)
top_order = np.argsort(corpus_counts)[::-1]

for index in top_order[:5]:
    print(lexical_words[index], int(corpus_counts[index]))


پردازش 2
من 2
را 2
دارم 2
زبان 2


## Tiny embeddings

Real embeddings may have hundreds or thousands of dimensions. Here we use only three so the
numbers are easy to inspect.


In [9]:
embedding_words = np.array(["زبان", "فارسی", "پردازش", "متن", "رایانه"])
embeddings = np.array([
    [0.90, 0.80, 0.10],
    [0.85, 0.75, 0.15],
    [0.70, 0.20, 0.65],
    [0.72, 0.25, 0.60],
    [0.10, 0.05, 0.95],
], dtype=np.float32)

print(embeddings.shape)


(5, 3)


## Normalize all embedding rows at once


In [10]:
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
normalized_embeddings = embeddings / norms

print(np.linalg.norm(normalized_embeddings, axis=1))


[0.99999994 1.         1.         0.99999994 0.99999994]


## Cosine-similarity matrix

After row normalization, a matrix multiplication gives every pairwise cosine similarity.


In [11]:
similarity_matrix = normalized_embeddings @ normalized_embeddings.T
print(np.round(similarity_matrix, 3))


[[1.    0.999 0.725 0.775 0.195]
 [0.999 1.    0.755 0.802 0.242]
 [0.725 0.755 1.    0.997 0.747]
 [0.775 0.802 0.997 1.    0.705]
 [0.195 0.242 0.747 0.705 1.   ]]


## Find nearest words


In [12]:
query_word = "زبان"
query_index = int(np.where(embedding_words == query_word)[0][0])
similarities = similarity_matrix[query_index]
ranked_indices = np.argsort(similarities)[::-1]

for index in ranked_indices[1:4]:  # skip the query itself
    print(embedding_words[index], float(similarities[index]))


فارسی 0.9988075494766235
متن 0.7747072577476501
پردازش 0.7250320315361023


## Sentence embedding by averaging word vectors

This is a simple baseline, not a replacement for contextual transformers.


In [13]:
embedding_lookup = {
    word: vector for word, vector in zip(embedding_words.tolist(), embeddings)
}

selected_sentence = ["پردازش", "متن"]
selected_vectors = np.stack([embedding_lookup[word] for word in selected_sentence])
sentence_embedding = selected_vectors.mean(axis=0)

print(sentence_embedding)


[0.71000004 0.225      0.625     ]


## Important warning

Do not store raw variable-length Python lists inside a NumPy `object` array for numerical ML
work. Pad them, truncate them, or keep them as Python lists until a framework-specific data
loader handles them.


## Tiny checkpoint

Count how many real tokens each padded sentence contains.


In [14]:
sequence_lengths = attention_mask.sum(axis=1)
print(sequence_lengths)


[6 5 6]
